# da 신뢰/비신뢰 리뷰 + SAM 보정 (Anki 스타일, 6-앙상블 기반)

새로 수집한 raw 프레임에 6-앙상블(신규 부트스트랩 5개 + 기존 배포모델, PROGRESS.md §2.20)로
da를 추론해서 오버레이로 보여주고, 사람이 **키보드 1(신뢰)/2(버리기)**로 빠르게 판단.

- **1(신뢰)**: 앙상블 예측을 그대로 채택하고 다음 프레임으로.
- **2(비신뢰)**: 그냥 버리지 않고 **SAM2 클릭 보정 모드로 즉시 전환** — 포함/제외 점을
  찍어서 그 프레임만 다시 라벨링. 다 됐으면 저장, 그래도 안 되겠으면 완전히 버림.
  SAM 연동은 최윤성님이 만든 `kuac_lane_prelabel_shared.ipynb`(SAM2+YOLOPv2 pre-labeling,
  2026-08-11 커밋)의 포인트 클릭(포함/제외) → 마스크 생성 → 수락 흐름을 참고해서 반영함
  (다만 그 노트북은 Colab+facebookresearch/sam2 레포 클론 방식이고, 여기서는 이미 로컬에
  있는 `sam2.1_b.pt` + `ultralytics.SAM` 래퍼로 동일한 멀티포인트 프롬프트를 구현 — 별도
  레포 클론/설치 없이 바로 됨).
- **주의(PROGRESS.md §2.9)**: SAM을 **자동/무보정으로** da·ll에 쓰는 건 이미 실패로
  결론남(벽/바닥까지 번짐, 완벽한 GT를 점 하나로 프롬프트해도 IoU가 오히려 떨어짐).
  여기서는 그거랑 다르게 **사람이 여러 점을 직접 찍고 눈으로 확인하며 반복 조정**하는
  거라 성격이 다름 — 그래도 검증된 적은 없는 시도이니, 실제로 잘 되는지는 써보면서
  판단할 것.
- 신뢰/보정 프레임 둘 다 `trust_review_output/trusted/`에 images+da_masks로 export
  (ll은 범위 밖 — 병합 전에 YOLOPv2+skeleton으로 별도 생성할 것)
- 진행 중 끊겨도 `trust_results.csv` + 마스크 캐시(`mask_cache/`)에 계속 저장되니
  다시 열면 이어서 진행됨
- 로컬 실행 전제(이 레포 루트에서 실행, `.venv_yolo` 커널) — 앙상블/SAM 모델이 이미
  로컬에 있어서 Colab/Drive 안 씀. 팀원과 같이 하고 싶으면 `demo.launch(share=True)`가
  만드는 공개 링크를 공유하면 됨.

## -1. 최초 1회 설정 (이 레포 처음 받은 사람만 — 이미 돌려봤으면 건너뛰기)

레포를 새로 clone했을 때 없는 것들만 자동으로 채워준다:
- Python 패키지(gradio/ultralytics/onnxruntime/opencv/torch) 설치
- `_TwinLiteNetPlus_ref/`(모델 아키텍처 코드) — 없으면 원본 저장소에서 clone
- `sam2.1_b.pt`(SAM2.1 체크포인트, 154MB) — GitHub 100MB 제한 때문에 레포에 직접
  못 올려서 [릴리즈 에셋](https://github.com/mastic-choi/TwinLiteNet-KMU-finetune/releases/tag/sam2-checkpoint)으로
  따로 배포함, `gh` CLI로 자동 다운로드(사전에 `gh auth login` 한 번 해뒀어야 함 —
  이 레포가 private라서 그냥 URL로는 안 받아짐)
- 앙상블 모델(`outputs/ensemble_bootstrap_v1/`)은 19MB로 작아서 레포에 이미 커밋돼
  있음 — clone만 하면 바로 있어야 정상, 없으면 git pull 다시 확인할 것

In [ ]:
import os, subprocess, sys

BASE = os.getcwd()

print("[설정] pip 패키지 설치 중...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                 "gradio", "ultralytics", "onnxruntime", "opencv-python", "torch", "numpy"], check=True)

ref_dir = os.path.join(BASE, "_TwinLiteNetPlus_ref")
if not os.path.isdir(ref_dir):
    print("[설정] _TwinLiteNetPlus_ref 없음 -> clone")
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/chequanghuy/TwinLiteNetPlus.git", ref_dir], check=True)
else:
    print("[설정] _TwinLiteNetPlus_ref 이미 있음 - 건너뜀")

sam_ckpt = os.path.join(BASE, "sam2.1_b.pt")
if not os.path.isfile(sam_ckpt):
    print("[설정] sam2.1_b.pt 없음 -> 릴리즈 에셋에서 다운로드 (gh CLI 필요, private repo라 인증 필요)")
    r = subprocess.run(["gh", "release", "download", "sam2-checkpoint",
                         "-R", "mastic-choi/TwinLiteNet-KMU-finetune",
                         "-p", "sam2.1_b.pt", "-D", BASE], capture_output=True, text=True)
    if r.returncode != 0:
        print("[설정] gh로 다운로드 실패 - 아래 참고해서 수동으로 받을 것:")
        print("  1) gh auth login (아직 안 했으면)")
        print("  2) gh release download sam2-checkpoint -R mastic-choi/TwinLiteNet-KMU-finetune")
        print("  stderr:", r.stderr)
    else:
        print("[설정] sam2.1_b.pt 다운로드 완료")
else:
    print("[설정] sam2.1_b.pt 이미 있음 - 건너뜀")

ensemble_dir = os.path.join(BASE, "outputs", "ensemble_bootstrap_v1")
missing = [f"seed{i}" for i in range(5)
           if not os.path.isfile(os.path.join(ensemble_dir, f"seed{i}", "best.pth"))]
if missing:
    print(f"[설정] 경고: 앙상블 모델 {missing} 없음 - git pull로 최신 상태인지 확인할 것")
else:
    print("[설정] 앙상블 모델 5개 전부 확인됨")

print("\n[설정] 완료 - 이제 아래 셀부터 순서대로 실행하면 됨")

## 0. 경로 설정 — INPUT_DIR을 실제 수집된 신규 프레임 폴더로 바꿀 것

In [2]:
import os, sys, csv
from argparse import Namespace

import cv2
import numpy as np
import onnxruntime as ort
import torch

BASE = os.getcwd()  # 이 노트북을 fine-tune 레포 루트에서 실행한다고 가정

# ↓↓↓ TODO: 실제 신규 수집 프레임 폴더로 바꿀 것 (예: 지그재그 주행 raw 프레임) ↓↓↓
INPUT_DIR = os.path.join(BASE, "new_raw_frames")
OUTPUT_DIR = os.path.join(BASE, "trust_review_output")
RESULTS_CSV = os.path.join(OUTPUT_DIR, "trust_results.csv")
MASK_CACHE_DIR = os.path.join(OUTPUT_DIR, "mask_cache")  # 최종 채택된 da 마스크 즉시 저장(크래시 대비)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MASK_CACHE_DIR, exist_ok=True)

ENSEMBLE_DIR = os.path.join(BASE, "outputs", "ensemble_bootstrap_v1")
DEPLOYED_ONNX = os.path.join(BASE, "outputs", "models", "best.onnx")
SAM_CKPT = os.path.join(BASE, "sam2.1_b.pt")
N_SEEDS = 5
CONFIG = "medium"
IN_W, IN_H = 640, 384

sys.path.insert(0, os.path.join(BASE, "_TwinLiteNetPlus_ref"))
from model.model import TwinLiteNetPlus

assert os.path.isdir(INPUT_DIR), f"{INPUT_DIR} 없음 - 신규 수집 프레임 폴더로 바꿀 것"
image_files = sorted(f for f in os.listdir(INPUT_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg")))
print(f"대상 이미지 {len(image_files)}장 ({INPUT_DIR})")

대상 이미지 925장 (/Users/mastic-choi/code/fine-tune/new_raw_frames)


## 1. 6-앙상블 모델 로드 (신규 부트스트랩 5개 + 기존 배포모델 1개)

In [3]:
torch_models = []
for seed in range(N_SEEDS):
    m = TwinLiteNetPlus(Namespace(config=CONFIG))
    state = torch.load(os.path.join(ENSEMBLE_DIR, f"seed{seed}", "best.pth"), map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    m.load_state_dict(state)
    m.eval()
    torch_models.append(m)

onnx_sess = ort.InferenceSession(DEPLOYED_ONNX, providers=["CPUExecutionProvider"])
print(f"6-앙상블 로드 완료 (신규 부트스트랩 {N_SEEDS}개 + 기존 배포모델 1개)")

6-앙상블 로드 완료 (신규 부트스트랩 5개 + 기존 배포모델 1개)


## 2. da 소프트보팅 추론 + 오버레이

In [4]:
def softmax_fg_np(l):
    m = l.max(axis=0, keepdims=True)
    e = np.exp(l - m)
    return (e / e.sum(axis=0, keepdims=True))[1]


def to_blob(img0):
    r = cv2.resize(img0, (IN_W, IN_H))
    rgb = cv2.cvtColor(r, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return np.ascontiguousarray(np.transpose(rgb, (2, 0, 1))[None, ...])


def infer_da_ensemble(img0):
    h, w = img0.shape[:2]
    blob_np = to_blob(img0)
    blob_t = torch.from_numpy(blob_np).float()
    probs = []
    for m in torch_models:
        with torch.no_grad():
            out_da, _ = m(blob_t)
        probs.append(torch.softmax(out_da, dim=1)[0, 1].numpy())
    da_out, _ = onnx_sess.run(["da", "ll"], {"images": blob_np})
    probs.append(softmax_fg_np(da_out[0]))
    avg = np.mean(probs, axis=0)
    return cv2.resize(avg, (w, h), interpolation=cv2.INTER_LINEAR) >= 0.5


def overlay_da(img_bgr, da_mask):
    vis = img_bgr.copy()
    vis[da_mask] = (0.55 * vis[da_mask].astype(np.float64) + 0.45 * np.array([0, 200, 0])).astype(np.uint8)
    return cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

## 3. SAM2 로드 (비신뢰 프레임 클릭 보정용)

최윤성님의 `kuac_lane_prelabel_shared.ipynb`와 같은 SAM2.1 계열이지만, 여기서는
`facebookresearch/sam2` 레포를 따로 클론하지 않고 이미 로컬에 있는 `sam2.1_b.pt`를
`ultralytics.SAM` 래퍼로 로드 — 멀티 포인트(포함/제외 혼합) 프롬프트 동일하게 지원됨.

In [5]:
from ultralytics import SAM

assert os.path.isfile(SAM_CKPT), f"{SAM_CKPT} 없음"
sam_model = SAM(SAM_CKPT)
print("SAM2(ultralytics wrapper) 로드 완료:", SAM_CKPT)


def draw_points_overlay(img_bgr, points, mask=None):
    vis = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy()
    if mask is not None:
        overlay = vis.copy()
        overlay[mask] = (0, 255, 0)
        vis = cv2.addWeighted(overlay, 0.4, vis, 0.6, 0)
    for x, y, lbl in points:
        color = (0, 200, 255) if lbl == 1 else (255, 0, 0)
        cv2.circle(vis, (int(x), int(y)), 6, color, -1)
    return vis


def run_sam(img_bgr, points):
    """points의 점 전부를 '한 오브젝트'에 대한 결합 프롬프트로 SAM에 넘긴다.
    주의: ultralytics.SAM은 points를 평평한 (N,2) 리스트로 주면 점마다 독립된
    오브젝트로 취급해서 각자 따로 세그멘테이션해버린다(포함/제외 아무 의미 없이
    첫 점 결과만 계속 나오는 버그의 원인이었음) - 반드시 (1, N, 2) 형태로 점
    전체를 하나의 오브젝트로 묶어서 줘야 포함/제외 점이 실제로 함께 반영된
    마스크 1개가 나온다."""
    if not points:
        return None
    pts = [[[p[0], p[1]] for p in points]]
    lbls = [[p[2] for p in points]]
    res = sam_model(img_bgr, points=pts, labels=lbls, verbose=False)
    r = res[0]
    if r.masks is None or len(r.masks.data) == 0:
        return None
    best_idx = int(torch.argmax(r.boxes.conf)) if r.boxes is not None and len(r.boxes.conf) else 0
    return r.masks.data[best_idx].cpu().numpy().astype(bool)

SAM2(ultralytics wrapper) 로드 완료: /Users/mastic-choi/code/fine-tune/sam2.1_b.pt


## 4. 상태/이어하기 — `trust_results.csv`/`mask_cache` 있으면 자동 이어하기.
여러 명이 동시에 공개 링크로 들어와도 서로 다른 프레임을 받도록 "찜(claim)" 방식을 씀:
누군가 프레임을 보기 시작하면 `claimed`에 표시되고, 그 프레임은 판정(신뢰/보정/버림)되거나
10분 넘게 방치되기 전까지 다른 사람에게 배정되지 않음.

In [6]:
import threading
import time

decisions = {}  # fname -> "trust" | "corrected" | "discard" (모든 세션이 공유)
if os.path.isfile(RESULTS_CSV):
    with open(RESULTS_CSV) as f:
        for row in csv.DictReader(f):
            decisions[row["file"]] = row["decision"]
    print(f"기존 진행상황 {len(decisions)}건 불러옴")

print(f"전체 {len(image_files)}장 / 완료 {len(decisions)}장 / 남은 {len(image_files) - len(decisions)}장")

_lock = threading.Lock()
claimed = {}  # fname -> claim된 시각(epoch) - "지금 누군가 보고 있음" 표시, 다른 세션에는 안 나감
CLAIM_TIMEOUT_SEC = 600  # 10분 넘게 확정 안 되면 방치된 걸로 보고 다른 사람이 다시 가져갈 수 있게 반납
_da_cache = {}  # fname -> 채택된 da 마스크(앙상블 or SAM보정), export 단계에서도 씀


def _is_available(fname):
    if fname in decisions:
        return False
    ts = claimed.get(fname)
    return ts is None or (time.time() - ts) > CLAIM_TIMEOUT_SEC


def _claim_next():
    """아직 아무도 안 보고 있는(또는 타임아웃 지난) 다음 프레임을 이 세션 몫으로 찜한다.
    여러 명이 동시에 접속해도 서로 다른 프레임을 받게 되는 핵심 로직 - 락으로 보호해서
    두 세션이 동시에 같은 프레임을 찜하는 경쟁 상태를 막는다."""
    with _lock:
        for fname in image_files:
            if _is_available(fname):
                claimed[fname] = time.time()
                return fname
        return None


def _release_claim(fname):
    with _lock:
        claimed.pop(fname, None)


def _load_review_overlay(fname):
    if fname is None:
        return None
    img = cv2.imread(os.path.join(INPUT_DIR, fname))
    if fname not in _da_cache:
        _da_cache[fname] = infer_da_ensemble(img)
    return overlay_da(img, _da_cache[fname])


def _status_for(fname):
    n_trust = sum(1 for v in decisions.values() if v == "trust")
    n_corrected = sum(1 for v in decisions.values() if v == "corrected")
    n_discard = sum(1 for v in decisions.values() if v == "discard")
    total, done = len(image_files), len(decisions)
    n_others = len(claimed) - (1 if fname in claimed else 0)
    if fname is None:
        return f"더 처리할 프레임 없음! 전체 완료 {done}/{total} (신뢰:{n_trust} 보정:{n_corrected} 버림:{n_discard})"
    return (f"{fname}  |  전체 {done}/{total} (신뢰:{n_trust} 보정:{n_corrected} 버림:{n_discard})"
            f"  |  동시 작업중인 다른 사람: {n_others}명")


def _finalize(fname, decision, mask=None):
    """최종 판정 1건을 기록: CSV에 append + (있으면) 마스크를 mask_cache에 즉시 저장 + claim 반납."""
    decisions[fname] = decision
    is_new = not os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.writer(f)
        if is_new:
            w.writerow(["file", "decision"])
        w.writerow([fname, decision])
    if mask is not None:
        _da_cache[fname] = mask
        cv2.imwrite(os.path.join(MASK_CACHE_DIR, fname), (mask.astype(np.uint8) * 255))
    _release_claim(fname)

남은 925장 / 전체 925장


## 5. Gradio 앱 — 리뷰(1=신뢰/2=비신뢰→SAM 보정), 세션별 프레임 자동 배정

`gr.State()`로 접속자(브라우저 세션)마다 독립된 진행 상태를 가짐 — 두 명이 동시에
들어와도 서로 다른 프레임을 자동 배정받고, 한쪽이 보고 있는 프레임은 판정 전까지
다른 쪽에 안 나감(§4의 `claimed` 방식).

In [7]:
import gradio as gr

KEY_JS = """
() => {
  if (window._daTrustKeyBound) { return; }
  window._daTrustKeyBound = true;

  function deepFindById(root, id) {
    if (!root) { return null; }
    if (root.getElementById) {
      const found = root.getElementById(id);
      if (found) { return found; }
    }
    const all = root.querySelectorAll ? root.querySelectorAll(\'*\') : [];
    for (const el of all) {
      if (el.id === id) { return el; }
      if (el.shadowRoot) {
        const found = deepFindById(el.shadowRoot, id);
        if (found) { return found; }
      }
    }
    return null;
  }

  function clickById(id) {
    const el = deepFindById(document, id);
    if (el) { (el.querySelector(\'button\') || el).click(); return true; }
    return false;
  }

  document.addEventListener(\'keydown\', function(e) {
    const tag = (e.target && e.target.tagName) || \'\';
    if (tag === \'INPUT\' || tag === \'TEXTAREA\') { return; }
    if (e.key === \'1\') {
      clickById(\'trust_btn\');
    } else if (e.key === \'2\') {
      clickById(\'discard_btn\');
    }
  });
}
"""


def on_load():
    fname = _claim_next()
    return (fname, "review", [], None,
            gr.update(visible=True), gr.update(visible=False),
            _load_review_overlay(fname), _status_for(fname))


def trust(sess_fname, sess_mode):
    if sess_mode != "review" or sess_fname is None:
        return sess_fname, sess_mode, gr.update(), gr.update(), gr.update(), gr.update()
    _finalize(sess_fname, "trust", mask=_da_cache.get(sess_fname))
    new_fname = _claim_next()
    return (new_fname, "review",
            gr.update(visible=True), gr.update(visible=False),
            _load_review_overlay(new_fname), _status_for(new_fname))


def enter_correction(sess_fname, sess_mode):
    if sess_mode != "review" or sess_fname is None:
        return sess_fname, sess_mode, [], None, gr.update(), gr.update(), gr.update(), gr.update()
    img = cv2.imread(os.path.join(INPUT_DIR, sess_fname))
    return (sess_fname, "correct", [], None,
            gr.update(visible=False), gr.update(visible=True),
            draw_points_overlay(img, []),
            f"[보정] {sess_fname} — 클릭으로 포함(초록점)/제외(빨간점) 표시 후 \'마스크 생성\'")


def on_click_point(sess_fname, sess_mode, sess_points, sess_mask, point_mode, evt: gr.SelectData):
    if sess_mode != "correct" or sess_fname is None:
        return sess_points, gr.update()
    x, y = evt.index
    label = 1 if point_mode == "포함 (da)" else 0
    sess_points = sess_points + [(x, y, label)]
    img = cv2.imread(os.path.join(INPUT_DIR, sess_fname))
    return sess_points, draw_points_overlay(img, sess_points, sess_mask)


def gen_mask(sess_fname, sess_mode, sess_points):
    if sess_mode != "correct" or sess_fname is None:
        return None, gr.update()
    img = cv2.imread(os.path.join(INPUT_DIR, sess_fname))
    mask = run_sam(img, sess_points)
    return mask, draw_points_overlay(img, sess_points, mask)


def reset_points(sess_fname, sess_mode):
    if sess_mode != "correct" or sess_fname is None:
        return [], None, gr.update()
    img = cv2.imread(os.path.join(INPUT_DIR, sess_fname))
    return [], None, draw_points_overlay(img, [])


def accept_correction(sess_fname, sess_mode, sess_mask):
    if sess_mode != "correct" or sess_fname is None:
        return sess_fname, sess_mode, [], None, gr.update(), gr.update(), gr.update(), gr.update()
    if sess_mask is not None:
        _finalize(sess_fname, "corrected", mask=sess_mask)
    else:
        _finalize(sess_fname, "discard")
    new_fname = _claim_next()
    return (new_fname, "review", [], None,
            gr.update(visible=True), gr.update(visible=False),
            _load_review_overlay(new_fname), _status_for(new_fname))


def give_up(sess_fname, sess_mode):
    if sess_mode != "correct" or sess_fname is None:
        return sess_fname, sess_mode, [], None, gr.update(), gr.update(), gr.update(), gr.update()
    _finalize(sess_fname, "discard")
    new_fname = _claim_next()
    return (new_fname, "review", [], None,
            gr.update(visible=True), gr.update(visible=False),
            _load_review_overlay(new_fname), _status_for(new_fname))


with gr.Blocks() as demo:
    gr.Markdown("### da 신뢰/비신뢰 리뷰 — **1(신뢰)** / **2(비신뢰→SAM 보정)**  "
                "| 여러 명이 동시에 접속해도 서로 다른 프레임이 배정됨")
    status = gr.Textbox(label="진행상황", interactive=False)

    # 세션(브라우저 탭)마다 독립된 상태 - 전역 dict로 하면 접속자 전원이 같은 화면을 보게 됨
    sess_fname = gr.State(None)
    sess_mode = gr.State("review")
    sess_points = gr.State([])
    sess_mask = gr.State(None)

    with gr.Group(visible=True) as review_group:
        review_img = gr.Image(interactive=False)
        with gr.Row():
            discard_btn = gr.Button("\u2717 비신뢰 -> SAM 보정 (2)", elem_id="discard_btn")
            trust_btn = gr.Button("\u2713 신뢰 (1)", elem_id="trust_btn", variant="primary")

    with gr.Group(visible=False) as correct_group:
        gr.Markdown("클릭한 점: 초록=포함(da) / 빨강=제외(배경) — 이미지 클릭으로 점 추가")
        point_mode = gr.Radio(["포함 (da)", "제외 (배경/콘)"], value="포함 (da)", label="다음 클릭 모드")
        correct_img = gr.Image(interactive=True)
        with gr.Row():
            gen_btn = gr.Button("마스크 생성")
            reset_btn = gr.Button("점 초기화")
            giveup_btn = gr.Button("그래도 버리기")
            accept_btn = gr.Button("보정 저장하고 다음 ->", variant="primary")

    demo.load(on_load, outputs=[sess_fname, sess_mode, sess_points, sess_mask,
                                 review_group, correct_group, review_img, status])
    trust_btn.click(trust, inputs=[sess_fname, sess_mode],
                     outputs=[sess_fname, sess_mode, review_group, correct_group, review_img, status])
    discard_btn.click(enter_correction, inputs=[sess_fname, sess_mode],
                       outputs=[sess_fname, sess_mode, sess_points, sess_mask,
                                review_group, correct_group, correct_img, status])
    correct_img.select(on_click_point,
                        inputs=[sess_fname, sess_mode, sess_points, sess_mask, point_mode],
                        outputs=[sess_points, correct_img])
    gen_btn.click(gen_mask, inputs=[sess_fname, sess_mode, sess_points], outputs=[sess_mask, correct_img])
    reset_btn.click(reset_points, inputs=[sess_fname, sess_mode], outputs=[sess_points, sess_mask, correct_img])
    accept_btn.click(accept_correction, inputs=[sess_fname, sess_mode, sess_mask],
                      outputs=[sess_fname, sess_mode, sess_points, sess_mask,
                               review_group, correct_group, review_img, status])
    giveup_btn.click(give_up, inputs=[sess_fname, sess_mode],
                      outputs=[sess_fname, sess_mode, sess_points, sess_mask,
                               review_group, correct_group, review_img, status])

demo.launch(share=True, debug=False, js=KEY_JS, inline=False)

/Users/mastic-choi/code/fine-tune/.venv_yolo/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/mastic-choi/code/fine-tune/.venv_yolo/lib/python3.13/site-packages/gradio/components/html.py:208: UserWarning: A `<script>` tag was found in the content of a `gr.HTML` component. Browsers do not execute `<script>` tags inserted via `innerHTML`, so this script will not run. Use the `head` parameter to load external libraries (e.g. `head='<script src="..."></script>'`); `head` content is injected and loaded before `js_on_load` runs. Then, if needed, put code that uses those libraries in the `js_on_load` parameter, which executes when the component renders. See https://gradio.app/guides/custom-HTML-components for details.
  _warn_if_script_tag(value)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://29ec04ea5dd0a2f735.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 6. Export — 신뢰/보정 처리된 프레임만 images/da_masks로 저장

`bootstrap_v2`와 같은 폴더 구조(`images/`, `da_masks/`)로 나오니, 그대로 복사해서
사람검증 corpus에 병합하면 됨. **ll_masks는 여기서 안 만듦** — 병합 전에
`scripts/pseudo_label/build_pseudo_label_dataset_v2.py`류의 YOLOPv2+skeleton
로직으로 따로 생성할 것. 순수 "버림(discard)" 처리된 프레임은 export 안 됨(데이터로 안 씀).

In [ ]:
EXPORT_DIR = os.path.join(OUTPUT_DIR, "trusted")
os.makedirs(os.path.join(EXPORT_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(EXPORT_DIR, "da_masks"), exist_ok=True)

n = 0
for fname, decision in decisions.items():
    if decision not in ("trust", "corrected"):
        continue
    img = cv2.imread(os.path.join(INPUT_DIR, fname))
    da_mask = _da_cache.get(fname)
    if da_mask is None:
        mask_cache_path = os.path.join(MASK_CACHE_DIR, fname)
        if os.path.isfile(mask_cache_path):
            da_mask = cv2.imread(mask_cache_path, cv2.IMREAD_GRAYSCALE) > 127
        else:
            da_mask = infer_da_ensemble(img)  # 폴백(정상 흐름에서는 안 탐)
    cv2.imwrite(os.path.join(EXPORT_DIR, "images", fname), img)
    cv2.imwrite(os.path.join(EXPORT_DIR, "da_masks", fname), (da_mask.astype(np.uint8) * 255))
    n += 1

n_trust = sum(1 for v in decisions.values() if v == "trust")
n_corrected = sum(1 for v in decisions.values() if v == "corrected")
print(f"export 완료: {n}장 (신뢰 {n_trust} + SAM보정 {n_corrected}) -> {EXPORT_DIR}")